## Creating the Dataset

The purpose of this notebook is to merge the two datasets together. Then curate the columns and set datatypes.

##### Data Sources
- Water temp data: https://library.ucsd.edu/dc/object/bb4003017c<br>
- Weather data (San Diego International Airport): https://www.ncei.noaa.gov/cdo-web/datasets/GHCND/stations/GHCND:USW00023188/detail <br>
- Buoy data - Station LJAC1 - 9410230: https://www.ndbc.noaa.gov/station_history.php?station=ljac1

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

In [7]:
weather_data_path = r'..\data\raw\4342131-NOAA-data-sd.csv'
pier_data_path = r'..\data\raw\pier-temp\LaJolla_TEMP_1916-202512.csv'

In [8]:
weather_df = pd.read_csv(weather_data_path)
water_df = pd.read_csv(pier_data_path,
                       header=46
                       )

In [10]:
len(weather_df.columns)

47

In [9]:
weather_df.head(2)

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,ACMH,ACSH,AWND,FMTM,...,WT09,WT10,WT11,WT13,WT14,WT16,WT18,WT21,WT22,WV01
0,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-01,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-02,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
water_df.head(2)

,YEAR,MONTH,DAY,TIME_PST,TIME_FLAG,SURF_TEMP_C,SURF_FLAG,BOT_TEMP_C,BOT_FLAG,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,1916.0,8.0,22.0,NaN,NaN,19.5,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1916.0,8.0,23.0,NaN,NaN,19.9,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 1. Create the date column for both dataframes

In [6]:
weather_df['date'] = pd.to_datetime(weather_df.DATE)

In [7]:
weather_df.head(2)

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,ACMH,ACSH,AWND,FMTM,...,WT10,WT11,WT13,WT14,WT16,WT18,WT21,WT22,WV01,date
0,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-01,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1939-07-01
1,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-02,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1939-07-02


For the water dataframe, it became apparent that there are some completely empty rows of data. Those are removed.

In [8]:
water_df.dropna(subset='YEAR', inplace=True)

In [9]:
water_df['YEAR'] = water_df.YEAR.astype(int).astype(str)
water_df['MONTH'] = water_df.MONTH.astype(int).astype(str)
water_df['DAY'] = water_df.DAY.astype(int).astype(str)

water_df['DATE'] = water_df.YEAR + '-' + water_df.MONTH + '-' + water_df.DAY

water_df['date'] = pd.to_datetime(water_df.DATE)

In [10]:
water_df.head(2)

,YEAR,MONTH,DAY,TIME_PST,TIME_FLAG,SURF_TEMP_C,SURF_FLAG,BOT_TEMP_C,BOT_FLAG,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,DATE,date
0,1916,8,22,NaN,NaN,19.5,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1916-8-22,1916-08-22
1,1916,8,23,NaN,NaN,19.9,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1916-8-23,1916-08-23


#### 2. Remove unneeded columns

In [11]:
list(water_df.columns)

['YEAR',
 'MONTH',
 'DAY',
 'TIME_PST',
 'TIME_FLAG',
 'SURF_TEMP_C',
 'SURF_FLAG',
 'BOT_TEMP_C',
 'BOT_FLAG',
 'Unnamed: 9',
 'Unnamed: 10',
 'Unnamed: 11',
 'Unnamed: 12',
 'Unnamed: 13',
 'DATE',
 'date']

In [12]:
drop_water_cols = ['Unnamed: 9',
 'Unnamed: 10',
 'Unnamed: 11',
 'Unnamed: 12',
 'Unnamed: 13',
 'DATE']
water_df.drop(columns=drop_water_cols, inplace=True)

In [13]:
water_df.head(2)

,YEAR,MONTH,DAY,TIME_PST,TIME_FLAG,SURF_TEMP_C,SURF_FLAG,BOT_TEMP_C,BOT_FLAG,date
0,1916,8,22,NaN,NaN,19.5,0.0,NaN,NaN,1916-08-22
1,1916,8,23,NaN,NaN,19.9,0.0,NaN,NaN,1916-08-23


To ensure every date is accounted for, a base_df is created with every date from the minimum available weather date, to the maximum available water temperature date.

In [ ]:
start_date = weather_df.date.min()
end_date = water_df.date.max().date()

# create the continuous index
date_index = pd.date_range(start=start_date, end=end_date)

In [15]:
base_df = pd.DataFrame(
    {'date': list(date_index)}
)

In [16]:
merge_1_df = base_df.merge(
    weather_df,
    how='left',
    on='date'
)

In [17]:
merge_2_df = merge_1_df.merge(water_df,
               on='date',
               how='left'
               )

In [18]:
merge_2_df.head(2)

,date,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,ACMH,ACSH,AWND,...,WV01,YEAR,MONTH,DAY,TIME_PST,TIME_FLAG,SURF_TEMP_C,SURF_FLAG,BOT_TEMP_C,BOT_FLAG
0,1939-07-01,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-01,NaN,NaN,NaN,...,NaN,1939,7,1,NaN,NaN,19.4,0.0,19.1,0.0
1,1939-07-02,USW00023188,"SAN DIEGO INTERNATIONAL AIRPORT, CA US",32.7336,-117.1831,4.6,1939-07-02,NaN,NaN,NaN,...,NaN,1939,7,2,NaN,NaN,19.8,0.0,19.5,0.0


#### 4. Curating the columns
##### Weather columns
There are many columns provided by the NOAA historical weather download. <br>
For details on all the available columns, please see the "weather-data-key.pdf" in the data folder.<br>

Listed below are the columns that were selected to keep in the dataframe.

date = Date of measurement
PRCP = Precipitation (mm or inches as per user preference, inches to hundredths on Daily Form pdf file) <br>
TMAX = Maximum temperature (Fahrenheit)<br>
TMIN = Minimum temperature (Fahrenheit)<br>
AWND = Average daily wind speed (meters per second or miles per hour as per user preference)<br>
<br>
BOT_TEMP_C = Water Temperature at the bottom of Scripp's pier <br>
SURF_TEMP_C = Water temperature at the surface (Target Variable)


In [27]:
final_colums = ['date','TMAX','TMIN','AWND', 'PRCP','BOT_TEMP_C', 'SURF_TEMP_C']

In [28]:
final_df = merge_2_df[final_colums].copy()

In [29]:
final_df.head(3)

,date,TMAX,TMIN,AWND,PRCP,BOT_TEMP_C,SURF_TEMP_C
0,1939-07-01,76.0,63.0,NaN,0.0,19.1,19.4
1,1939-07-02,74.0,65.0,NaN,0.0,19.5,19.8
2,1939-07-03,71.0,62.0,NaN,0.0,19.5,20.1


The data types are already in a desirable format so there is no need to set them.

In [31]:
final_df.dtypes

date           datetime64[us]
TMAX                  float64
TMIN                  float64
AWND                  float64
PRCP                  float64
BOT_TEMP_C            float64
SURF_TEMP_C           float64
dtype: object

### 5. Save merged data

In [30]:
final_df.to_parquet(r'..\data\raw\final-raw-merged-data.parquet',
                      engine='pyarrow',
                      index=False
                      )

---

### 6. Buoy Data

Here we will read in and partially clean the buoy data.

In [2]:
buoy_data_folder = Path('../data/raw/buoy-data-la-jolla')
if not buoy_data_folder.exists():
    raise ValueError('BUOY DATA folder path does not EXIST')

In [3]:
buoy_paths = [x for x in buoy_data_folder.glob('*.txt')]

In [4]:
def fix_year(year):
    if len(str(year)) == 2:
        if year < 30:
            new_year = 2000 + year
        else:
            new_year = 1900 + year
        return new_year
    return year

In [5]:
def read_buoy_texts(buoy_txt_path):
    """
    purpose: read in the buoy text files into dataframes. 
    """
    
    with open(buoy_txt_path) as t:
        first_line = t.readline()

    if first_line.startswith('#'):
        df = pd.read_csv(buoy_txt_path,
                         sep=r'\s+',
                         skiprows=[1],
                         header=0
                         )
        df.columns = [c.lstrip('#') for c in df.columns]
        
        return df
    df = pd.read_csv(buoy_txt_path,
                        sep=r'\s+',
                        header=0
                    )
    df.rename(columns={'WD': 'WDIR', 'YYYY': 'YY'}, inplace=True)
    if df.YY.max() < 100:
        df['YY'] = df.YY.apply(lambda x: fix_year(x))
    return df


In [6]:
buoy_dfs = [read_buoy_texts(path) for path in buoy_paths]

In [7]:
buoy_df = pd.concat(buoy_dfs)

In [8]:
buoy_df.head(1)

,YY,MM,DD,hh,mm,WDIR,WSPD,GST,WVHT,DPD,APD,MWD,BAR,ATMP,WTMP,DEWP,VIS,TIDE,PRES
0,2005,1,1,0,30,270,3.0,4.0,1.4,10.0,99.0,999,9999.0,13.3,999.0,999.0,99.0,99.0,NaN


Creating the date column

In [9]:
datetime_parts = dict(year=buoy_df.YY, month=buoy_df.MM, day=buoy_df.DD)
buoy_df['date'] = pd.to_datetime(datetime_parts)
buoy_df.head(1)

,YY,MM,DD,hh,mm,WDIR,WSPD,GST,WVHT,DPD,APD,MWD,BAR,ATMP,WTMP,DEWP,VIS,TIDE,PRES,date
0,2005,1,1,0,30,270,3.0,4.0,1.4,10.0,99.0,999,9999.0,13.3,999.0,999.0,99.0,99.0,NaN,2005-01-01


In [10]:
buoy_df.shape

(163662, 20)

In [11]:
cols_to_keep = [
    'date',
    'hh',
    'WDIR',
    'WSPD'
]

final_buoy_df = buoy_df[cols_to_keep].copy()

In [12]:
final_buoy_df.WDIR.describe()

count    163662.000000
mean        348.431432
std         328.342030
min           0.000000
25%         132.000000
50%         260.000000
75%         335.000000
max         999.000000
Name: WDIR, dtype: float64

In [13]:
final_buoy_df.WSPD.describe()

count    163662.000000
mean         20.076221
std          37.442327
min           0.000000
25%           1.200000
50%           2.600000
75%           5.000000
max          99.000000
Name: WSPD, dtype: float64

I am also just going to clean the data a bit. From the source it is known that wind speed of 99 or WDIR of 999 are null values.

In [14]:
final_buoy_df['WDIR'] = final_buoy_df.WDIR.replace({999: np.nan})
final_buoy_df['WSPD'] = final_buoy_df.WSPD.replace({99: np.nan})

In [15]:
final_buoy_df.WSPD.describe()

count    133640.000000
mean          2.346128
std           1.782122
min           0.000000
25%           1.000000
50%           2.000000
75%           3.200000
max          15.800000
Name: WSPD, dtype: float64

In [16]:
final_buoy_df['WDIR'].describe()

count    132903.000000
mean        197.864187
std         110.154874
min           0.000000
25%         108.000000
50%         222.000000
75%         295.000000
max         360.000000
Name: WDIR, dtype: float64

Now we can see we no longer have those outlier null indicator values. Next the data will be collapse to one row per day by taking the mean per day.

In [17]:
daily_mean_buoy_data = final_buoy_df.groupby(['date',])[[ 'WSPD', 'WDIR']].mean().reset_index()
daily_mean_buoy_data.head(1)

,date,WSPD,WDIR
0,2005-01-01,2.26087,185.0


In [18]:
daily_mean_buoy_data.shape

(7022, 3)

In [19]:
daily_mean_buoy_data.to_parquet('../data/raw/raw-daily-buoy-data.parquet',
                         index=False,
                         engine='pyarrow'
                         )

In [20]:
temp_data_df = pd.read_parquet(r'..\data\raw\final-raw-merged-data.parquet')

Merging the buoy data with the water and air temperature data.

In [21]:
merge_buoy_temp_data = daily_mean_buoy_data.merge(temp_data_df,
                                            how='left',
                                                on='date')

In [22]:
merge_buoy_temp_data.to_parquet('../data/raw/raw-buoy-water-weather-data.parquet',
                                engine='pyarrow',
                                index=False
                                )